# SAE feature-atlas walkthrough

This notebook runs one Geneformer layer end-to-end on a small, deterministic subsample so that reviewers can reproduce the pipeline without committing ~400 GB of disk. It assumes you have the following on disk already:

1. Geneformer V2-316M pretrained model (HuggingFace `ctheodoris/Geneformer`, subfolder `Geneformer-V2-316M`).
2. Replogle K562 CRISPRi data (`replogle_concat.h5ad` or the Figshare release).
3. The `bio_mech_interp` conda environment (see `requirements.txt`).

If you have the full phase-1 activation extractions cached under `experiments/phase1_k562/`, the notebook will pick them up and skip the forward-pass step.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import torch

PROJECT = Path('/Volumes/Crucial X6/MacBook/biomechinterp/biodyn-work/subproject_42_sparse_autoencoder_biological_map')
sys.path.insert(0, str(PROJECT / 'src'))
from sae_model import TopKSAE, SAETrainer

LAYER = 11
PHASE1 = PROJECT / 'experiments' / 'phase1_k562'
print('Project:', PROJECT)
print('Layer  :', LAYER)

## 1. Load a small sample of residual-stream activations

We work with 100K gene-position activations at the chosen layer. If the full 4M-position cache is already materialized under `experiments/phase1_k562/layer_{LAYER:02d}_activations.npy`, we deterministically sample from it; otherwise re-run `src/01_extract_activations.py` on a 200-cell subset first.

In [ ]:
act_path = PHASE1 / f'layer_{LAYER:02d}_activations.npy'
assert act_path.exists(), f'Missing {act_path}. Run src/01_extract_activations.py first.'
act = np.lib.format.open_memmap(str(act_path), mode='r')
print('full activations shape:', act.shape)
rng = np.random.default_rng(42)
idx = np.sort(rng.choice(act.shape[0], size=100_000, replace=False))
sample = np.asarray(act[idx], dtype=np.float32)
mean = sample.mean(axis=0)
print('sample:', sample.shape, 'mean norm:', float(np.linalg.norm(mean)))

## 2. Train a small TopK SAE

We use the smaller 2x expansion for speed. On an Apple M-series with MPS this completes in a minute or two; on CPU expect ~5 minutes.

In [ ]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
sae = TopKSAE(d_model=1152, n_features=2 * 1152, k=32)
trainer = SAETrainer(sae, lr=3e-4, device=device)
centered = sample - mean[None, :]
for epoch in range(3):
    loss = trainer.train_epoch(centered, batch_size=4096, log_every=10**7)
    print(f'epoch {epoch+1}: train mse = {loss:.4f}')

## 3. Inspect reconstruction and feature statistics

In [ ]:
sae.eval().to('cpu')
x = torch.tensor(sample[:10000] - mean[None, :], dtype=torch.float32)
with torch.no_grad():
    x_hat, h_sparse, top_idx = sae(x)
    total_var = x.var(dim=0).sum().item()
    resid_var = (x - x_hat).var(dim=0).sum().item()
    var_expl = 1.0 - resid_var / total_var
    act_freq = (h_sparse > 0).float().mean(dim=0)
    dead = int((act_freq == 0).sum())
print(f'variance explained : {var_expl:.3f}')
print(f'alive features     : {sae.n_features - dead}/{sae.n_features}')
print(f'dead features      : {dead}')

## 4. (optional) Run downstream analyses on the toy SAE

The full annotation / causal patching / perturbation pipeline is driven by `src/03_*.py` through `src/09_*.py`. To run any of them against the toy SAE, point their `SAE_BASE` / `run_name` variables at a temporary directory where you save `sae_final.pt` and `activation_mean.npy` (mirrored to the structure expected by the other scripts).

For the reviewer-requested ablations and controls added in this revision (SVD threshold sweep, Leiden resolution sweep, cross-layer PMI null, multi-database perturbation test, matched-input cross-model comparison, batch/assay confound analysis), see `src_revision/b1_svd_threshold_sweep.py` through `src_revision/c6_batch_assay_analysis.py` and the orchestration entry point `run_all.sh revision`.